<a href="https://colab.research.google.com/github/irritatednishant/next_word_predictor/blob/main/version1.0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

In [2]:
import tensorflow as tf

path_to_file = tf.keras.utils.get_file(
    'shakespeare.txt',
    'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt'
)


1115394/1115394 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
text = open(file=path_to_file, mode='r').read()


In [4]:
text = text.lower()
print(len(text))

1115394


In [5]:
text[:500]

"first citizen:\nbefore we proceed any further, hear me speak.\n\nall:\nspeak, speak.\n\nfirst citizen:\nyou are all resolved rather to die than to famish?\n\nall:\nresolved. resolved.\n\nfirst citizen:\nfirst, you know caius marcius is chief enemy to the people.\n\nall:\nwe know't, we know't.\n\nfirst citizen:\nlet us kill him, and we'll have corn at our own price.\nis't a verdict?\n\nall:\nno more talking on't; let it be done: away, away!\n\nsecond citizen:\none word, good citizens.\n\nfirst citizen:\nwe are accounted poor"

In [6]:
text = text[:500000]

In [7]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [8]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])

total_words = len(tokenizer.word_index) + 1
print(total_words)

8244


In [9]:
type(text)

str

In [10]:
input_sequence = []
for line in text.split('\n'):
  list_tokenize = tokenizer.texts_to_sequences([line])[0]

  for i in range(1, len(list_tokenize)):
    sequence = list_tokenize[:i+1]
    input_sequence.append(sequence)

In [11]:
input_sequence[0:6]

[[64, 142],
 [148, 31],
 [148, 31, 878],
 [148, 31, 878, 181],
 [148, 31, 878, 181, 438],
 [148, 31, 878, 181, 438, 131]]

In [12]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_len = max(len(seq) for seq in input_sequence)
input_sequence = pad_sequences(input_sequence, maxlen=max_len, padding='pre')

In [13]:
input_sequence.shape

(76514, 16)

In [14]:
input_sequence[0:6]

array([[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,  64, 142],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0, 148,  31],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
        148,  31, 878],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0, 148,
         31, 878, 181],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0, 148,  31,
        878, 181, 438],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0, 148,  31, 878,
        181, 438, 131]], dtype=int32)

In [15]:
import numpy as np

In [16]:
type(input_sequence)

numpy.ndarray

In [17]:
input_sequence = np.array(input_sequence)
X = input_sequence[:,0:-1]
y = input_sequence[:,-1]

In [18]:
split = int(0.8 * len(X))

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

In [19]:
print(X.shape)
print(X_train.shape)
print(y.shape)
print(y_train.shape)
print(max_len)
print(total_words)

(76514, 15)
(61211, 15)
(76514,)
(61211,)
16
8244


In [21]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

# Hardcoding total_words as it was undefined in the execution scope.
# In a complete workflow, ensure the cell defining total_words is run beforehand.
total_words = 12633
# Hardcoding max_len as it was undefined in the execution scope.
# In a complete workflow, ensure the cell defining max_len is run beforehand.
max_len = 16

model = Sequential()

model.add(
    Embedding(
        input_dim=total_words, # Using the hardcoded total_words to fix NameError
        output_dim=100
    ) # Removed input_length as it's deprecated
)

model.add(LSTM(150))


model.add(Dense(total_words, activation='softmax'))

# Explicitly build the model before calling summary
model.build(input_shape=(None, max_len - 1))

model.compile(
    loss='sparse_categorical_crossentropy', # Changed to sparse_categorical_crossentropy for integer labels
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 15, 100)        │     1,263,300 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 150)            │       150,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 12633)          │     1,907,583 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,321,483 (12.67 MB)

 Trainable params: 3,321,483 (12.67 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=120,
    batch_size=128,
    validation_data=(X_test, y_test)
)

Epoch 1/120
479/479 ━━━━━━━━━━━━━━━━━━━━ 79s 159ms/step - accuracy: 0.0342 - loss: 7.0289 - val_accuracy: 0.0375 - val_loss: 7.0074
Epoch 2/120
479/479 ━━━━━━━━━━━━━━━━━━━━ 76s 159ms/step - accuracy: 0.0470 - loss: 6.5331 - val_accuracy: 0.0457 - val_loss: 6.9642
Epoch 3/120
479/479 ━━━━━━━━━━━━━━━━━━━━ 80s 155ms/step - accuracy: 0.0592 - loss: 6.3292 - val_accuracy: 0.0531 - val_loss: 6.9218
Epoch 4/120
479/479 ━━━━━━━━━━━━━━━━━━━━ 74s 155ms/step - accuracy: 0.0746 - loss: 6.1266 - val_accuracy: 0.0671 - val_loss: 6.8597
Epoch 5/120
479/479 ━━━━━━━━━━━━━━━━━━━━ 74s 155ms/step - accuracy: 0.0924 - loss: 5.9249 - val_accuracy: 0.0729 - val_loss: 6.8495
Epoch 6/120
479/479 ━━━━━━━━━━━━━━━━━━━━ 73s 153ms/step - accuracy: 0.1029 - loss: 5.7541 - val_accuracy: 0.0817 - val_loss: 6.8589
Epoch 7/120
479/479 ━━━━━━━━━━━━━━━━━━━━ 83s 155ms/step - accuracy: 0.1097 - loss: 5.6065 - val_accuracy: 0.0833 - val_loss: 6.8905
Epoch 8/120
479/479 ━━━━━━━━━━━━━━━━━━━━ 74s 153ms/step - accuracy: 0.1147 -

In [ ]:
import numpy as np

def predict_next_word(seed_text):

    token_list = tokenizer.texts_to_sequences([seed_text])[0]

    token_list = pad_sequences(
        [token_list],
        maxlen=max_len-1,
        padding='pre'
    )

    predicted = np.argmax(
        model.predict(token_list, verbose=0),
        axis=-1
    )[0]

    for word, index in tokenizer.word_index.items():
        if index == predicted:
            return word

In [ ]:
print(f'Next word for "to": {predict_next_word("to")}')
print(f'Next word for "the day is": {predict_next_word("the day is")}')
print(f'Next word for "i will not": {predict_next_word("i will not")}')
print(f'Next word for "wherefore art thou": {predict_next_word("wherefore art thou")}')
print(f'Next word for "o, romeo, romeo": {predict_next_word("o, romeo, romeo")}')
print(f'Next word for "but soft, what light through yonder window": {predict_next_word("but soft, what light through yonder window")}')
print(f'Next word for "all the world s": {predict_next_word("all the world s")}')
print(f'Next word for "a rose by any other": {predict_next_word("a rose by any other")}')
print(f'Next word for "to be or not to": {predict_next_word("to be or not to")}')
print(f'Next word for "the quality of mercy is not": {predict_next_word("the quality of mercy is not")}')

In [ ]:
print('hello')